In [ ]:
import sys
import subprocess
sys.path.insert(0, '..')

try:
    import numdifftools  # noqa: F401
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'numdifftools'])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from scipy.stats import norm
warnings.filterwarnings('ignore')

T_WINDOW = 30
TAU_MIN = 3       # Minimum consecutive days below barrier for DI variant
BARRIER_LEVEL = 0.6  # Barrier as fraction of strike (e.g., 60% of ATM strike)
CUMULATIVE_THRESHOLD = 15.0  # Cumulative shortfall threshold for CB variant

# Exact imports from solarrpy
from solarrpy.buyer import (
    compute_buyer_table, unhedged_cf, benchmark_cf, hedged_cf,
    optimal_Nc, mape, E_f, N_p, E_bar, tick,
)
from solarrpy.sorad import daily_strike, sorad_price
from solarrpy.forecastDensity import clearsky_at, seasonal_mean_Y_at, conditional_moments_Y, R_to_Y, Y_to_R
from solarrpy.calibration import calibrate_window

# BSoRad (Barrier SoRad) pricing helpers

def barrier_di_price(
    K: float,
    R_t0: float,
    cal,
    dates,
    barrier: float | None = None,
    tau_min: int = 3,
    n_sim: int = 50_000,
    seed: int = 42,
) -> float:
    """
    Down-and-In Barrier SoRad put price via Monte Carlo.

    The contract activates (payoff = max(K - R_T, 0)) only if
    at least tau_min consecutive days have radiation < barrier.

    Payoff = max(K - R_T, 0) · tick · indicator_run

    Parameters
    ----------
    K        : float – put strike (ATM daily strike)
    R_t0     : float – last observed GHI at start of window
    cal             – calibrated model from calibrate_window
    dates    : array-like – dates in the window
    barrier  : float – knock-in barrier level; if None, uses BARRIER_LEVEL * K
    tau_min  : int – minimum consecutive days below barrier to trigger
    n_sim    : int – number of MC paths
    seed     : int – RNG seed for reproducibility
    """
    if barrier is None:
        barrier = BARRIER_LEVEL * K

    rng = np.random.default_rng(seed)
    T = len(dates)  # number of days in window

    # Get OU parameters
    mu = cal.mu
    sigma = cal.sigma
    theta = float(cal.theta)

    # Get seasonal parameters for initial state
    dates_pd = pd.to_datetime(dates)
    t_now = dates_pd[0]
    Y_t0 = R_to_Y(R_t0, clearsky_at(t_now, cal), cal.alpha, cal.beta)
    Y_bar_t0 = seasonal_mean_Y_at(t_now, cal)

    # Simulate OU paths: Y_n = Y_bar_n + (Y_t0 - Y_bar_t0) * exp(-theta * (n - t)) + noise
    # For simplicity, use a discretized OU with daily steps
    payoffs = np.zeros(n_sim)

    for sim in range(n_sim):
        Y_path = np.zeros(T)
        R_path = np.zeros(T)

        for day in range(T):
            t_day = dates_pd[day]
            C_day = clearsky_at(t_day, cal)
            Y_bar_day = seasonal_mean_Y_at(t_day, cal)
            decay = np.exp(-theta * (day + 1))  # OU mean reversion

            # Draw innovation
            dW = rng.standard_normal()
            Y_path[day] = Y_bar_day + (Y_t0 - Y_bar_t0) * decay + sigma * dW

            # Transform back to R-space
            R_path[day] = Y_to_R(Y_path[day], C_day, cal.alpha, cal.beta)

        # Check for consecutive days below barrier
        below_barrier = R_path < barrier
        max_consecutive = 0
        current_run = 0
        for is_below in below_barrier:
            if is_below:
                current_run += 1
                max_consecutive = max(max_consecutive, current_run)
            else:
                current_run = 0

        # Payoff: intrinsic on final day only if barrier was hit
        intrinsic = max(K - R_path[-1], 0.0)
        indicator = 1.0 if max_consecutive >= tau_min else 0.0
        payoffs[sim] = intrinsic * indicator

    return float(np.mean(payoffs) * tick)


def barrier_cb_price(
    K: float,
    R_t0: float,
    cal,
    dates,
    barrier: float | None = None,
    cumsum_threshold: float | None = None,
    n_sim: int = 50_000,
    seed: int = 42,
) -> float:
    """
    Cumulative Barrier SoRad put price via Monte Carlo.

    The contract activates if the cumulative shortfall
    sum((barrier - R_n)+) exceeds a threshold.

    Payoff = max(K - R_T, 0) · tick · indicator_cumsum

    Parameters
    ----------
    K        : float – put strike
    R_t0     : float – last observed GHI at start of window
    cal             – calibrated model from calibrate_window
    dates    : array-like – dates in the window
    barrier  : float – reference level for shortfall; if None, uses BARRIER_LEVEL * K
    cumsum_threshold : float – cumulative shortfall trigger; if None, uses CUMULATIVE_THRESHOLD
    n_sim    : int – number of MC paths
    seed     : int – RNG seed for reproducibility
    """
    if barrier is None:
        barrier = BARRIER_LEVEL * K
    if cumsum_threshold is None:
        cumsum_threshold = CUMULATIVE_THRESHOLD

    rng = np.random.default_rng(seed)
    T = len(dates)

    # Get OU parameters
    mu = cal.mu
    sigma = cal.sigma
    theta = float(cal.theta)

    # Initial conditions
    dates_pd = pd.to_datetime(dates)
    t_now = dates_pd[0]
    Y_t0 = R_to_Y(R_t0, clearsky_at(t_now, cal), cal.alpha, cal.beta)
    Y_bar_t0 = seasonal_mean_Y_at(t_now, cal)

    payoffs = np.zeros(n_sim)

    for sim in range(n_sim):
        Y_path = np.zeros(T)
        R_path = np.zeros(T)
        cumsum_shortfall = 0.0

        for day in range(T):
            t_day = dates_pd[day]
            C_day = clearsky_at(t_day, cal)
            Y_bar_day = seasonal_mean_Y_at(t_day, cal)
            decay = np.exp(-theta * (day + 1))

            # Draw innovation
            dW = rng.standard_normal()
            Y_path[day] = Y_bar_day + (Y_t0 - Y_bar_t0) * decay + sigma * dW

            # Transform back to R-space
            R_path[day] = Y_to_R(Y_path[day], C_day, cal.alpha, cal.beta)

            # Accumulate shortfall
            shortfall = max(barrier - R_path[day], 0.0)
            cumsum_shortfall += shortfall

        # Payoff: intrinsic on final day only if cumsum threshold exceeded
        intrinsic = max(K - R_path[-1], 0.0)
        indicator = 1.0 if cumsum_shortfall >= cumsum_threshold else 0.0
        payoffs[sim] = intrinsic * indicator

    return float(np.mean(payoffs) * tick)

In [ ]:
df = pd.read_csv('../data/Bologna.csv', parse_dates=['date'])
print(f'Full dataset: {len(df)} rows, years {df["Year"].min()}–{df["Year"].max()}')
print(f'Buyer params: E_f={E_f}, N_p={N_p}, E_bar={E_bar}, tick={tick}')
print(f'Barrier window T={T_WINDOW} days')
print(f'Barrier level: {BARRIER_LEVEL*100:.0f}% of strike, Min consecutive days: {TAU_MIN}')
print(f'Cumulative threshold: {CUMULATIVE_THRESHOLD:.1f}')

In [ ]:
def build_windows(df_year, T):
    """Split a yearly DataFrame into non-overlapping windows of T days."""
    df_year = df_year.reset_index(drop=True)
    windows = []
    n = len(df_year)
    for start in range(0, n - T + 1, T):
        chunk = df_year.iloc[start:start + T]
        windows.append(chunk)
    return windows


def barrier_unhedged_cf(R_bar, T):
    """Unhedged cash flow for the buyer over one window of T days."""
    return E_f * N_p * R_bar * E_bar * T


def barrier_benchmark_cf(K_B, T):
    """Benchmark (target) cash flow: revenue at strike level."""
    return E_f * N_p * K_B * E_bar * T


def barrier_hedged_cf_di(R_bar, K_B, V0_DI, Nc, T):
    """Hedged cash flow for the buyer with DI barrier option."""
    # Payoff is stochastic (depends on barrier hit), but we use intrinsic as proxy
    payoff_proxy = max(K_B - R_bar, 0.0)
    return (
        barrier_unhedged_cf(R_bar, T)
        + Nc * (payoff_proxy * tick - V0_DI * tick)
    )


def barrier_hedged_cf_cb(R_bar, K_B, V0_CB, Nc, T):
    """Hedged cash flow for the buyer with CB barrier option."""
    payoff_proxy = max(K_B - R_bar, 0.0)
    return (
        barrier_unhedged_cf(R_bar, T)
        + Nc * (payoff_proxy * tick - V0_CB * tick)
    )


def barrier_optimal_Nc_di(windows_data):
    """
    Closed-form optimal N_c for DI barrier option,
    minimising sum of squared deviations from benchmark cash flow.

    windows_data: list of dicts with keys R_bar, K_B, V0_DI, T
    """
    numerator = 0.0
    denominator = 0.0
    for w in windows_data:
        R_bar  = w['R_bar']
        K_B    = w['K_B']
        V0_DI  = w['V0_DI']
        T      = w['T']
        CF_u   = barrier_unhedged_cf(R_bar, T)
        CF_b   = barrier_benchmark_cf(K_B, T)
        payoff_proxy = max(K_B - R_bar, 0.0)
        delta  = (payoff_proxy - V0_DI) * tick
        numerator   += (CF_b - CF_u) * delta
        denominator += delta ** 2
    return numerator / denominator if denominator != 0 else 0.0


def barrier_optimal_Nc_cb(windows_data):
    """
    Closed-form optimal N_c for CB barrier option,
    minimising sum of squared deviations from benchmark cash flow.

    windows_data: list of dicts with keys R_bar, K_B, V0_CB, T
    """
    numerator = 0.0
    denominator = 0.0
    for w in windows_data:
        R_bar  = w['R_bar']
        K_B    = w['K_B']
        V0_CB  = w['V0_CB']
        T      = w['T']
        CF_u   = barrier_unhedged_cf(R_bar, T)
        CF_b   = barrier_benchmark_cf(K_B, T)
        payoff_proxy = max(K_B - R_bar, 0.0)
        delta  = (payoff_proxy - V0_CB) * tick
        numerator   += (CF_b - CF_u) * delta
        denominator += delta ** 2
    return numerator / denominator if denominator != 0 else 0.0


print('Helper functions defined.')

In [ ]:
print('Running Barrier SoRad buyer train-test loop 2014–2023...')
print('Pricing both DI (Down-and-In) and CB (Cumulative Barrier) variants...\n')

eval_years = list(range(2014, 2024))
records_di = []
records_cb = []

for eval_year in eval_years:
    # --- Training set: all years strictly before eval_year ---
    df_train = df[df['Year'] < eval_year].copy()
    first_train = df_train['Year'].min()
    print(f'  [{eval_year}] Training {first_train}–{eval_year - 1}... ', end='', flush=True)

    # Calibrate model on training data
    cal = calibrate_window(
        df_train,
        train_end_year=eval_year - 1,
        coords={'lat': 44.5},
    )

    # --- Evaluation set: the eval_year ---
    df_eval = df[df['Year'] == eval_year].copy()
    windows = build_windows(df_eval, T_WINDOW)

    windows_data_di = []
    windows_data_cb = []

    for win in windows:
        dates_w = win['date'].values
        R_vals  = win['GHI'].values
        R_bar_w = R_vals.mean()
        R_t0_w  = R_vals[0]  # Last observed GHI at start of window

        # ATM strike for this window (mean of daily strikes)
        dates_pd = pd.to_datetime(dates_w)
        K_B_w = float(np.mean([daily_strike(d, cal) for d in dates_pd]))

        # DI barrier price
        V0_DI_w = barrier_di_price(
            K=K_B_w,
            R_t0=R_t0_w,
            cal=cal,
            dates=dates_w,
            barrier=BARRIER_LEVEL * K_B_w,
            tau_min=TAU_MIN,
            n_sim=10_000,
            seed=42 + eval_year,
        )

        # CB barrier price
        V0_CB_w = barrier_cb_price(
            K=K_B_w,
            R_t0=R_t0_w,
            cal=cal,
            dates=dates_w,
            barrier=BARRIER_LEVEL * K_B_w,
            cumsum_threshold=CUMULATIVE_THRESHOLD,
            n_sim=10_000,
            seed=42 + eval_year,
        )

        windows_data_di.append({
            'R_bar': R_bar_w,
            'K_B':   K_B_w,
            'V0_DI': V0_DI_w,
            'T':     T_WINDOW,
        })

        windows_data_cb.append({
            'R_bar': R_bar_w,
            'K_B':   K_B_w,
            'V0_CB': V0_CB_w,
            'T':     T_WINDOW,
        })

    # Optimal N_c for both variants
    Nc_di = barrier_optimal_Nc_di(windows_data_di)
    Nc_cb = barrier_optimal_Nc_cb(windows_data_cb)

    # MAPE and seller net for DI
    CF_u_list_di = [barrier_unhedged_cf(w['R_bar'], w['T']) for w in windows_data_di]
    CF_b_list_di = [barrier_benchmark_cf(w['K_B'],  w['T']) for w in windows_data_di]
    CF_h_list_di = [
        barrier_hedged_cf_di(w['R_bar'], w['K_B'], w['V0_DI'], Nc_di, w['T'])
        for w in windows_data_di
    ]
    MAPE_u_di = 100.0 * np.mean(
        [abs(CF_u - CF_b) / abs(CF_b) for CF_u, CF_b in zip(CF_u_list_di, CF_b_list_di)]
    )
    MAPE_h_di = 100.0 * np.mean(
        [abs(CF_h - CF_b) / abs(CF_b) for CF_h, CF_b in zip(CF_h_list_di, CF_b_list_di)]
    )
    seller_net_di = sum(
        Nc_di * (w['V0_DI'] - max(w['K_B'] - w['R_bar'], 0.0)) * tick
        for w in windows_data_di
    )

    # MAPE and seller net for CB
    CF_u_list_cb = [barrier_unhedged_cf(w['R_bar'], w['T']) for w in windows_data_cb]
    CF_b_list_cb = [barrier_benchmark_cf(w['K_B'],  w['T']) for w in windows_data_cb]
    CF_h_list_cb = [
        barrier_hedged_cf_cb(w['R_bar'], w['K_B'], w['V0_CB'], Nc_cb, w['T'])
        for w in windows_data_cb
    ]
    MAPE_u_cb = 100.0 * np.mean(
        [abs(CF_u - CF_b) / abs(CF_b) for CF_u, CF_b in zip(CF_u_list_cb, CF_b_list_cb)]
    )
    MAPE_h_cb = 100.0 * np.mean(
        [abs(CF_h - CF_b) / abs(CF_b) for CF_h, CF_b in zip(CF_h_list_cb, CF_b_list_cb)]
    )
    seller_net_cb = sum(
        Nc_cb * (w['V0_CB'] - max(w['K_B'] - w['R_bar'], 0.0)) * tick
        for w in windows_data_cb
    )

    print(f'DI: N_c={Nc_di:.1f}  MAPE_h={MAPE_h_di:.2f}%  seller={seller_net_di:.2f}€ | '
          f'CB: N_c={Nc_cb:.1f}  MAPE_h={MAPE_h_cb:.2f}%  seller={seller_net_cb:.2f}€')

    records_di.append({
        'eval_year':      eval_year,
        'n_windows':      len(windows_data_di),
        'N_c_opt':        Nc_di,
        'unhedged_MAPE':  MAPE_u_di,
        'hedged_MAPE':    MAPE_h_di,
        'seller_net_eur': seller_net_di,
        '_windows_data':  windows_data_di,
        '_CF_u':          CF_u_list_di,
        '_CF_b':          CF_b_list_di,
        '_CF_h':          CF_h_list_di,
    })

    records_cb.append({
        'eval_year':      eval_year,
        'n_windows':      len(windows_data_cb),
        'N_c_opt':        Nc_cb,
        'unhedged_MAPE':  MAPE_u_cb,
        'hedged_MAPE':    MAPE_h_cb,
        'seller_net_eur': seller_net_cb,
        '_windows_data':  windows_data_cb,
        '_CF_u':          CF_u_list_cb,
        '_CF_b':          CF_b_list_cb,
        '_CF_h':          CF_h_list_cb,
    })

table_di = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in records_di])
table_cb = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in records_cb])
print('\nDone.')

In [ ]:
print('\n' + '='*100)
print('Table: Buyer/Seller Analysis — Down-and-In Barrier SoRad (Bologna, 2014–2023)')
print('='*100)
display_cols = ['eval_year', 'n_windows', 'N_c_opt', 'unhedged_MAPE', 'hedged_MAPE', 'seller_net_eur']
print(table_di[display_cols].to_string(index=False, float_format='{:.3f}'.format))

print(f'\nHedged MAPE < Unhedged MAPE: {(table_di["hedged_MAPE"] < table_di["unhedged_MAPE"]).sum()}/10 years')
print(f'Mean unhedged MAPE: {table_di["unhedged_MAPE"].mean():.2f}%')
print(f'Mean hedged MAPE:   {table_di["hedged_MAPE"].mean():.2f}%')
print(f'Mean seller net:    {table_di["seller_net_eur"].mean():.2f} €/year')

table_di.to_csv('../results/BarrierDI_Buyer.csv', index=False, float_format='%.6f')
print('\nSaved to results/BarrierDI_Buyer.csv')

In [ ]:
print('\n' + '='*100)
print('Table: Buyer/Seller Analysis — Cumulative Barrier SoRad (Bologna, 2014–2023)')
print('='*100)
display_cols = ['eval_year', 'n_windows', 'N_c_opt', 'unhedged_MAPE', 'hedged_MAPE', 'seller_net_eur']
print(table_cb[display_cols].to_string(index=False, float_format='{:.3f}'.format))

print(f'\nHedged MAPE < Unhedged MAPE: {(table_cb["hedged_MAPE"] < table_cb["unhedged_MAPE"]).sum()}/10 years')
print(f'Mean unhedged MAPE: {table_cb["unhedged_MAPE"].mean():.2f}%')
print(f'Mean hedged MAPE:   {table_cb["hedged_MAPE"].mean():.2f}%')
print(f'Mean seller net:    {table_cb["seller_net_eur"].mean():.2f} €/year')

table_cb.to_csv('../results/BarrierCB_Buyer.csv', index=False, float_format='%.6f')
print('\nSaved to results/BarrierCB_Buyer.csv')

In [ ]:
# Collect mean prices per year
mean_V0_DI = [np.mean([w['V0_DI'] for w in r['_windows_data']]) for r in records_di]
mean_V0_CB = [np.mean([w['V0_CB'] for w in r['_windows_data']]) for r in records_cb]
years = table_di['eval_year'].values

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Top-left: DI mean price
ax = axes[0, 0]
ax.bar(years, mean_V0_DI, color='#FF6B6B', alpha=0.8, edgecolor='white')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Mean ATM DI Barrier put price', fontsize=11)
ax.set_title('DI Barrier: Mean Premium', fontsize=12)
ax.set_xticks(years)
ax.set_xticklabels(years, rotation=45, ha='right')
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Top-right: CB mean price
ax = axes[0, 1]
ax.bar(years, mean_V0_CB, color='#4ECDC4', alpha=0.8, edgecolor='white')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Mean ATM CB Barrier put price', fontsize=11)
ax.set_title('CB Barrier: Mean Premium', fontsize=12)
ax.set_xticks(years)
ax.set_xticklabels(years, rotation=45, ha='right')
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Bottom-left: DI MAPE comparison
ax = axes[1, 0]
x = np.arange(len(years))
width = 0.35
ax.bar(x - width/2, table_di['unhedged_MAPE'].values, width,
        label='Unhedged MAPE', color='#2196F3', alpha=0.85, edgecolor='white')
ax.bar(x + width/2, table_di['hedged_MAPE'].values, width,
        label='Hedged MAPE', color='#FF9800', alpha=0.85, edgecolor='white')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('MAPE (%)', fontsize=11)
ax.set_title('DI Barrier: Unhedged vs Hedged MAPE', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(years, rotation=45, ha='right')
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Bottom-right: CB MAPE comparison
ax = axes[1, 1]
ax.bar(x - width/2, table_cb['unhedged_MAPE'].values, width,
        label='Unhedged MAPE', color='#2196F3', alpha=0.85, edgecolor='white')
ax.bar(x + width/2, table_cb['hedged_MAPE'].values, width,
        label='Hedged MAPE', color='#FF9800', alpha=0.85, edgecolor='white')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('MAPE (%)', fontsize=11)
ax.set_title('CB Barrier: Unhedged vs Hedged MAPE', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(years, rotation=45, ha='right')
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.5)

fig.suptitle(
    f'Barrier SoRad (T={T_WINDOW} days) — Bologna 2014–2023',
    fontsize=14, fontweight='bold', y=0.995
)
plt.tight_layout()
plt.savefig('../results/Figure_Barrier_Prices_MAPE.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/Figure_Barrier_Prices_MAPE.png')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# DI seller P&L
seller_pnl_di = table_di['seller_net_eur'].values
colors_di = ['#4CAF50' if v >= 0 else '#F44336' for v in seller_pnl_di]
ax = axes[0]
ax.bar(years, seller_pnl_di, color=colors_di, alpha=0.85, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axhline(
    table_di['seller_net_eur'].mean(), color='navy', linewidth=1.5,
    linestyle=':', label=f'Mean = {table_di["seller_net_eur"].mean():.0f} €'
)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Seller net P&L (€)', fontsize=11)
ax.set_title('DI Barrier: Seller Net P&L', fontsize=12)
ax.set_xticks(years)
ax.set_xticklabels(years, rotation=45, ha='right')
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)

# CB seller P&L
seller_pnl_cb = table_cb['seller_net_eur'].values
colors_cb = ['#4CAF50' if v >= 0 else '#F44336' for v in seller_pnl_cb]
ax = axes[1]
ax.bar(years, seller_pnl_cb, color=colors_cb, alpha=0.85, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axhline(
    table_cb['seller_net_eur'].mean(), color='navy', linewidth=1.5,
    linestyle=':', label=f'Mean = {table_cb["seller_net_eur"].mean():.0f} €'
)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Seller net P&L (€)', fontsize=11)
ax.set_title('CB Barrier: Seller Net P&L', fontsize=12)
ax.set_xticks(years)
ax.set_xticklabels(years, rotation=45, ha='right')
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)

fig.suptitle(
    f'Barrier SoRad Seller Net P&L — Bologna 2014–2023',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('../results/Figure_Barrier_SellerPnL.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/Figure_Barrier_SellerPnL.png')

In [ ]:
# Example: final year, both DI and CB
last_rec_di  = records_di[-1]
last_rec_cb  = records_cb[-1]
last_year = last_rec_di['eval_year']
n_win     = len(last_rec_di['_CF_u'])
win_idx   = np.arange(1, n_win + 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

# DI cash flows
ax = axes[0]
ax.plot(win_idx, last_rec_di['_CF_b'], 'k--', lw=1.8, label='Benchmark $CF_b$')
ax.plot(win_idx, last_rec_di['_CF_u'], 'b-o', lw=1.4, ms=5, alpha=0.8, label='Unhedged $CF_u$')
ax.plot(win_idx, last_rec_di['_CF_h'], 'r-s', lw=1.4, ms=5, alpha=0.8, label='Hedged $CF_h$ (DI)')
ax.set_xlabel(f'Window index ({T_WINDOW}-day blocks)', fontsize=11)
ax.set_ylabel('Cash flow (€)', fontsize=11)
ax.set_title(
    f'DI Barrier Cash Flows — {last_year}\n'
    f'$N_c$ = {last_rec_di["N_c_opt"]:.1f}, MAPE_h = {last_rec_di["hedged_MAPE"]:.2f}%',
    fontsize=12
)
ax.legend(fontsize=10)
ax.grid(linestyle='--', alpha=0.4)

# CB cash flows
ax = axes[1]
ax.plot(win_idx, last_rec_cb['_CF_b'], 'k--', lw=1.8, label='Benchmark $CF_b$')
ax.plot(win_idx, last_rec_cb['_CF_u'], 'b-o', lw=1.4, ms=5, alpha=0.8, label='Unhedged $CF_u$')
ax.plot(win_idx, last_rec_cb['_CF_h'], 'r-s', lw=1.4, ms=5, alpha=0.8, label='Hedged $CF_h$ (CB)')
ax.set_xlabel(f'Window index ({T_WINDOW}-day blocks)', fontsize=11)
ax.set_ylabel('Cash flow (€)', fontsize=11)
ax.set_title(
    f'CB Barrier Cash Flows — {last_year}\n'
    f'$N_c$ = {last_rec_cb["N_c_opt"]:.1f}, MAPE_h = {last_rec_cb["hedged_MAPE"]:.2f}%',
    fontsize=12
)
ax.legend(fontsize=10)
ax.grid(linestyle='--', alpha=0.4)

fig.suptitle(
    f'Barrier SoRad Cash-Flow Comparison — Bologna {last_year}',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('../results/Figure_Barrier_CashFlows_Example.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/Figure_Barrier_CashFlows_Example.png')